# Experiment 2 - Effect of Input Size

Runs the same pipeline (encrypt, add, multiply, sum, average, decrypt) at vector sizes 5, 10, 50, 100 and 500, and reports how the runtime of each of the three implementations changes with the amount of data.

Not included in the final report, which cites the same effect from the healthcare experiment instead. Kept here as the direct measurement of it.

## 1. Install and load the project

In [ ]:
!pip install -q tenseal openfhe numpy matplotlib

In [ ]:
import os
import subprocess
import sys

REPO = "https://github.com/To2004/confidential-computing-project.git"

# Works both on a fresh Colab runtime and inside a local checkout.
if not os.path.exists("src/benchmark_harness.py"):
    if not os.path.exists("confidential-computing-project"):
        subprocess.run(["git", "clone", "-q", REPO], check=True)
    os.chdir("confidential-computing-project")

# Absolute, so imports survive a later change of directory, and guarded so
# re-running this cell does not stack duplicate entries.
SRC = os.path.abspath("src")
if SRC not in sys.path:
    sys.path.insert(0, SRC)

# Write this notebook's output to its own directory, so running it does not
# overwrite the results and figures the report was built from.
import benchmark_harness as harness
import project_paths

os.makedirs("notebook_output", exist_ok=True)
project_paths.RESULTS_DIR = "notebook_output"
project_paths.FIGURES_DIR = "notebook_output"

print("working directory:", os.getcwd())
print("report uses", harness.DEFAULT_REPEATS, "repetitions per measurement")

## 2. Check which libraries loaded

The OpenFHE wheel ships a binary built for CPython 3.8. On a newer Python it
installs but fails to import.

Without OpenFHE the experiment still runs, with the plaintext and TenSEAL columns only.

In [ ]:
import importlib

for name in ("tenseal", "openfhe"):
    try:
        importlib.import_module(name)
        print(f"{name}: available")
    except Exception as exc:
        print(f"{name}: not available -- {exc}")

## 3. Run the experiment

`REPEATS` is the number of timed repetitions per measurement. The report uses 1000 on a reserved compute node; this notebook uses fewer so it finishes in a few minutes. The numbers it prints will therefore be noisier than the ones in the report.

In [ ]:
REPEATS = 20

!python src/benchmark_scaling.py --repeats {REPEATS} --warmup 5 --output notebook_output/scaling_results.json

## 4. Results

In [ ]:
import plot_results as pr
from IPython.display import Image, display

pr.apply_style()
pr.chart_scaling(pr.load("scaling_results.json"))
display(Image("notebook_output/chart_scaling.png"))